In [25]:
import os
import zipfile
import gdown
from pathlib import Path

def download_and_extract():
    # Google Drive File ID from the link:
    # https://drive.google.com/file/d/1oMDFwlGwObPg0kmyFVGVoYEMtc8ubOWT/view
    file_id = "1oMDFwlGwObPg0kmyFVGVoYEMtc8ubOWT"
    url = f"https://drive.google.com/uc?id={file_id}"

    # Target directories
    raw_dir = Path("data/raw")
    raw_dir.mkdir(parents=True, exist_ok=True)

    zip_path = raw_dir / "rdd2022_india.zip"

    # Download
    print(f"Downloading dataset from Google Drive (ID: {file_id})...")
    try:
        gdown.download(url, str(zip_path), quiet=False)
    except Exception as e:
        print(f"Error downloading via gdown: {e}")
        print("Please check if gdown is installed and the Google Drive link is accessible.")
        return

    # Extract
    if zip_path.exists():
        print("Extracting dataset...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(raw_dir)
            print("Extraction completed successfully!")

            # Clean up the zip file after extraction
            os.remove(zip_path)
            print("Cleaned up the zip file.")
        except Exception as e:
            print(f"Error during extraction: {e}")
    else:
        print("Download failed, zip file not found.")

if __name__ == "__main__":
    download_and_extract()


Downloading...
From (original): https://drive.google.com/uc?id=1oMDFwlGwObPg0kmyFVGVoYEMtc8ubOWT
From (redirected): https://drive.google.com/uc?id=1oMDFwlGwObPg0kmyFVGVoYEMtc8ubOWT&confirm=t&uuid=485737c6-8cdb-4bd7-81be-6cf698864088
To: /content/image-preprocessing-classification/data/raw/rdd2022_india.zip
100%|██████████| 527M/527M [00:05<00:00, 93.5MB/s]


Extracting dataset...
Extraction completed successfully!
Cleaned up the zip file.


In [26]:
import os
import xml.etree.ElementTree as ET
import cv2
import random
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

CLASSES = ["D00", "D20", "D40"]

def get_iou(boxA, boxB):
    # box format: [xmin, ymin, xmax, ymax]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    if boxAArea + boxBArea - interArea == 0:
        return 0

    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

def check_overlap_with_any(candidate_box, ground_truth_boxes):
    for gt_box in ground_truth_boxes:
        if get_iou(candidate_box, gt_box) > 0.0:
            return True
    return False

def parse_xml(xml_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception as e:
        print(f"Error parsing XML {xml_path}: {e}")
        return None, []

    filename = root.find("filename").text

    # Get image size
    size_node = root.find("size")
    width = int(size_node.find("width").text)
    height = int(size_node.find("height").text)

    objects = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name in CLASSES:
            bndbox = obj.find("bndbox")
            xmin = int(float(bndbox.find("xmin").text))
            ymin = int(float(bndbox.find("ymin").text))
            xmax = int(float(bndbox.find("xmax").text))
            ymax = int(float(bndbox.find("ymax").text))

            # Make sure coordinates are within image boundaries
            xmin = max(0, xmin)
            ymin = max(0, ymin)
            xmax = min(width, xmax)
            ymax = min(height, ymax)

            if xmax > xmin and ymax > ymin:
                objects.append({
                    "class": name,
                    "box": [xmin, ymin, xmax, ymax]
                })

    return filename, objects

def process_dataset():
    raw_dir = Path("data/raw")
    processed_dir = Path("data/processed")

    # Locate images and annotations
    # RDD2022 India zip usually contains India/images/ and India/annotations/xmls/
    india_dir = None
    for p in raw_dir.glob("**/annotations/xmls"):
        india_dir = p.parent.parent
        break

    if not india_dir:
        print("Could not find India annotations directory. Please check extraction path.")
        return

    images_dir = india_dir / "images"
    annotations_dir = india_dir / "annotations" / "xmls"

    print(f"Found dataset at: {india_dir}")
    xml_files = list(annotations_dir.glob("*.xml"))
    print(f"Total xml annotation files: {len(xml_files)}")

    # Split xml files at image level to avoid data leakage
    train_xml, test_xml = train_test_split(xml_files, test_size=0.3, random_state=42)
    val_xml, test_xml = train_test_split(test_xml, test_size=0.5, random_state=42)

    splits = {
        "train": train_xml,
        "val": val_xml,
        "test": test_xml
    }

    # Initialize output class directories
    for split in splits:
        for cls in CLASSES + ["Normal"]:
            (processed_dir / split / cls).mkdir(parents=True, exist_ok=True)

    # Process each split
    for split_name, files in splits.items():
        print(f"Processing {split_name} split ({len(files)} images)...")

        for xml_path in tqdm(files):
            img_filename, objects = parse_xml(xml_path)
            if img_filename is None:
                continue

            img_path = images_dir / img_filename
            # Fallback in case of extensions mismatches
            if not img_path.exists():
                img_path = images_dir / (xml_path.stem + ".jpg")

            if not img_path.exists():
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                continue

            h, w, _ = img.shape
            gt_boxes = [obj["box"] for obj in objects]

            # 1. Crop actual damage patches
            for idx, obj in enumerate(objects):
                cls_name = obj["class"]
                box = obj["box"]
                xmin, ymin, xmax, ymax = box

                # Crop and save patch
                patch = img[ymin:ymax, xmin:xmax]
                if patch.size > 0:
                    patch_name = f"{xml_path.stem}_damage_{idx}_{cls_name}.jpg"
                    out_path = processed_dir / split_name / cls_name / patch_name
                    cv2.imwrite(str(out_path), patch)

            # 2. Sample random Normal patches
            # Try to crop a few non-overlapping background patches of average damage size (e.g. 128x128)
            patch_size = 128
            sampled_normal = 0
            # Limit the number of normal patches per image to keep classes balanced
            max_normal_per_image = max(1, len(objects))

            attempts = 0
            while sampled_normal < max_normal_per_image and attempts < 20:
                attempts += 1
                if w - patch_size <= 0 or h - patch_size <= 0:
                    break
                nx = random.randint(0, w - patch_size)
                ny = random.randint(0, h - patch_size)
                candidate_box = [nx, ny, nx + patch_size, ny + patch_size]

                if not check_overlap_with_any(candidate_box, gt_boxes):
                    patch = img[ny:ny+patch_size, nx:nx+patch_size]
                    if patch.size > 0:
                        patch_name = f"{xml_path.stem}_normal_{sampled_normal}.jpg"
                        out_path = processed_dir / split_name / "Normal" / patch_name
                        cv2.imwrite(str(out_path), patch)
                        sampled_normal += 1

    print("Patch preprocessing completed!")
    # Show stats
    for split in splits:
        print(f"\nStats for {split}:")
        for cls in CLASSES + ["Normal"]:
            path = processed_dir / split / cls
            count = len(list(path.glob("*.jpg")))
            print(f"  - {cls}: {count} patches")

if __name__ == "__main__":
    process_dataset()


Found dataset at: data/raw/India/train
Total xml annotation files: 7706
Processing train split (5394 images)...


100%|██████████| 5394/5394 [00:17<00:00, 304.50it/s]


Processing val split (1156 images)...


100%|██████████| 1156/1156 [00:03<00:00, 327.78it/s]


Processing test split (1156 images)...


100%|██████████| 1156/1156 [00:04<00:00, 273.78it/s]


Patch preprocessing completed!

Stats for train:
  - D00: 1110 patches
  - D20: 1434 patches
  - D40: 2219 patches
  - Normal: 7889 patches

Stats for val:
  - D00: 197 patches
  - D20: 280 patches
  - D40: 476 patches
  - Normal: 1641 patches

Stats for test:
  - D00: 248 patches
  - D20: 307 patches
  - D40: 492 patches
  - Normal: 1708 patches


In [44]:
!pwd
!ls -F

/content/image-preprocessing-classification
configs/	 data/	     outputs/	requirements.txt
CONTRIBUTING.md  notebooks/  README.md	src/


In [45]:
!python src/preprocessing/transforms.py

In [46]:
!python src/models/baseline_cnn.py

In [48]:
!python src/training/dataset.py

## **0. Baseline:**

In [49]:
!python -m src.data.preprocess_dataset --config configs/baseline.yaml


--- Offline Preprocessing: baseline ---
Reading raw cropped images from: data/processed/cropped
Saving preprocessed images to: data/processed/baseline
Processing split 'train', class 'D00' (1110 images)...
100% 1110/1110 [00:00<00:00, 1110.53it/s]
Processing split 'train', class 'D20' (1434 images)...
100% 1434/1434 [00:01<00:00, 770.07it/s]
Processing split 'train', class 'D40' (2219 images)...
100% 2219/2219 [00:01<00:00, 1816.91it/s]
Processing split 'train', class 'Normal' (7889 images)...
100% 7889/7889 [00:04<00:00, 1625.36it/s]
Processing split 'val', class 'D00' (197 images)...
100% 197/197 [00:00<00:00, 1793.12it/s]
Processing split 'val', class 'D20' (280 images)...
100% 280/280 [00:00<00:00, 1128.70it/s]
Processing split 'val', class 'D40' (476 images)...
100% 476/476 [00:00<00:00, 1664.94it/s]
Processing split 'val', class 'Normal' (1641 images)...
100% 1641/1641 [00:01<00:00, 1579.62it/s]
Processing split 'test', class 'D00' (248 images)...
100% 248/248 [00:00<00:00, 1633

In [50]:
# Baseline – Train
!python -m src.training.train --config configs/baseline.yaml

--- Starting Experiment: baseline ---
Using device: cuda
[train] Loading preprocessed dataset from: data/processed/baseline/train
Loaded 12652 images for split: train
[val] Loading preprocessed dataset from: data/processed/baseline/val
Loaded 2594 images for split: val
Using CosineAnnealingLR scheduler (T_max=30, eta_min=1e-06)
Epoch [1/30] LR: 0.001000 | Train Loss: 0.6950 | Train Acc: 70.14% | Val Loss: 0.7245 | Val Acc: 70.28%
  => Saved new best model checkpoint (Val Acc: 70.28%)
Epoch [2/30] LR: 0.000997 | Train Loss: 0.6443 | Train Acc: 72.46% | Val Loss: 0.6515 | Val Acc: 72.74%
  => Saved new best model checkpoint (Val Acc: 72.74%)
Epoch [3/30] LR: 0.000989 | Train Loss: 0.6236 | Train Acc: 73.40% | Val Loss: 0.6013 | Val Acc: 74.40%
  => Saved new best model checkpoint (Val Acc: 74.40%)
Epoch [4/30] LR: 0.000976 | Train Loss: 0.6087 | Train Acc: 74.11% | Val Loss: 0.5640 | Val Acc: 77.06%
  => Saved new best model checkpoint (Val Acc: 77.06%)
Epoch [5/30] LR: 0.000957 | Train 

In [32]:
!ls outputs/baseline

In [52]:
# Baseline – Evaluate on test set
!python -m src.evaluation.evaluate --config configs/baseline.yaml --model_path outputs/baseline/best_model.pth

--- Evaluating model for: baseline ---
[test] Loading preprocessed dataset from: data/processed/baseline/test
Loaded 2755 images for split: test
Model weights loaded successfully.

--- Test Set Classification Report ---
              precision    recall  f1-score   support

         D00       0.78      0.66      0.72       248
         D20       0.79      0.79      0.79       307
         D40       0.86      0.87      0.86       492
      Normal       0.95      0.97      0.96      1708

    accuracy                           0.90      2755
   macro avg       0.85      0.82      0.83      2755
weighted avg       0.90      0.90      0.90      2755

Overall Accuracy: 90.31%

Evaluation completed. Metrics and confusion matrix saved to outputs/baseline


## **1. Letterbox:**

In [37]:
!python -m src.data.preprocess_dataset --config configs/pipeline1_letterbox.yaml


--- Offline Preprocessing: pipeline1_letterbox ---
Reorganizing raw cropped patches: moving train/val/test to data/processed/cropped
Reading raw cropped images from: data/processed/cropped
Saving preprocessed images to: data/processed/letterbox
Processing split 'train', class 'D00' (1110 images)...
100% 1110/1110 [00:00<00:00, 1578.30it/s]
Processing split 'train', class 'D20' (1434 images)...
100% 1434/1434 [00:01<00:00, 747.27it/s]
Processing split 'train', class 'D40' (2219 images)...
100% 2219/2219 [00:01<00:00, 1456.18it/s]
Processing split 'train', class 'Normal' (7889 images)...
100% 7889/7889 [00:05<00:00, 1389.18it/s]
Processing split 'val', class 'D00' (197 images)...
100% 197/197 [00:00<00:00, 1454.82it/s]
Processing split 'val', class 'D20' (280 images)...
100% 280/280 [00:00<00:00, 799.39it/s]
Processing split 'val', class 'D40' (476 images)...
100% 476/476 [00:00<00:00, 1587.11it/s]
Processing split 'val', class 'Normal' (1641 images)...
100% 1641/1641 [00:01<00:00, 1577

In [39]:
# Pipeline 1 – Train (Letterbox Resize)
!python -m src.training.train --config configs/pipeline1_letterbox.yaml

--- Starting Experiment: pipeline1_letterbox ---
Using device: cuda
[train] Loading preprocessed dataset from: data/processed/letterbox/train
Loaded 12652 images for split: train
[val] Loading preprocessed dataset from: data/processed/letterbox/val
Loaded 2594 images for split: val
Using CosineAnnealingLR scheduler (T_max=30, eta_min=1e-06)
Epoch [1/30] LR: 0.001000 | Train Loss: 0.3892 | Train Acc: 83.73% | Val Loss: 0.3689 | Val Acc: 84.73%
  => Saved new best model checkpoint (Val Acc: 84.73%)
Epoch [2/30] LR: 0.000997 | Train Loss: 0.3181 | Train Acc: 86.25% | Val Loss: 0.2915 | Val Acc: 87.43%
  => Saved new best model checkpoint (Val Acc: 87.43%)
Epoch [3/30] LR: 0.000989 | Train Loss: 0.3004 | Train Acc: 87.43% | Val Loss: 0.3097 | Val Acc: 87.05%
Epoch [4/30] LR: 0.000976 | Train Loss: 0.2920 | Train Acc: 88.11% | Val Loss: 0.2650 | Val Acc: 88.86%
  => Saved new best model checkpoint (Val Acc: 88.86%)
Epoch [5/30] LR: 0.000957 | Train Loss: 0.3151 | Train Acc: 87.13% | Val Los

In [40]:
# Pipeline 1 – Evaluate  test set
!python -m src.evaluation.evaluate --config configs/pipeline1_letterbox.yaml --model_path outputs/pipeline1_letterbox/best_model.pth

--- Evaluating model for: pipeline1_letterbox ---
[test] Loading preprocessed dataset from: data/processed/letterbox/test
Loaded 2755 images for split: test
Model weights loaded successfully.

--- Test Set Classification Report ---
              precision    recall  f1-score   support

         D00       0.72      0.58      0.64       248
         D20       0.68      0.38      0.49       307
         D40       0.63      0.87      0.73       492
      Normal       1.00      1.00      1.00      1708

    accuracy                           0.87      2755
   macro avg       0.76      0.70      0.71      2755
weighted avg       0.87      0.87      0.86      2755

Overall Accuracy: 86.79%

Evaluation completed. Metrics and confusion matrix saved to outputs/pipeline1_letterbox


## **2. CLAHE:**

In [38]:
!python -m src.data.preprocess_dataset --config configs/pipeline2_clahe.yaml


--- Offline Preprocessing: pipeline2_clahe ---
Reading raw cropped images from: data/processed/cropped
Saving preprocessed images to: data/processed/clahe
Processing split 'train', class 'D00' (1110 images)...
100% 1110/1110 [00:02<00:00, 520.77it/s]
Processing split 'train', class 'D20' (1434 images)...
100% 1434/1434 [00:03<00:00, 387.65it/s]
Processing split 'train', class 'D40' (2219 images)...
100% 2219/2219 [00:02<00:00, 878.35it/s]
Processing split 'train', class 'Normal' (7889 images)...
100% 7889/7889 [00:11<00:00, 695.23it/s]
Processing split 'val', class 'D00' (197 images)...
100% 197/197 [00:00<00:00, 809.45it/s]
Processing split 'val', class 'D20' (280 images)...
100% 280/280 [00:00<00:00, 404.89it/s]
Processing split 'val', class 'D40' (476 images)...
100% 476/476 [00:00<00:00, 846.63it/s]
Processing split 'val', class 'Normal' (1641 images)...
100% 1641/1641 [00:02<00:00, 742.48it/s]
Processing split 'test', class 'D00' (248 images)...
100% 248/248 [00:00<00:00, 810.57i

In [41]:
# Pipeline 2 – Train (CLAHE)
!python -m src.training.train --config configs/pipeline2_clahe.yaml

--- Starting Experiment: pipeline2_clahe ---
Using device: cuda
[train] Loading preprocessed dataset from: data/processed/clahe/train
Loaded 12652 images for split: train
[val] Loading preprocessed dataset from: data/processed/clahe/val
Loaded 2594 images for split: val
Using CosineAnnealingLR scheduler (T_max=30, eta_min=1e-06)
Epoch [1/30] LR: 0.001000 | Train Loss: 0.5910 | Train Acc: 76.75% | Val Loss: 1.0650 | Val Acc: 79.72%
  => Saved new best model checkpoint (Val Acc: 79.72%)
Epoch [2/30] LR: 0.000997 | Train Loss: 0.4443 | Train Acc: 83.57% | Val Loss: 0.4370 | Val Acc: 85.58%
  => Saved new best model checkpoint (Val Acc: 85.58%)
Epoch [3/30] LR: 0.000989 | Train Loss: 0.4101 | Train Acc: 85.09% | Val Loss: 0.4474 | Val Acc: 83.96%
Epoch [4/30] LR: 0.000976 | Train Loss: 0.3606 | Train Acc: 87.09% | Val Loss: 0.4463 | Val Acc: 83.62%
Epoch [5/30] LR: 0.000957 | Train Loss: 0.3526 | Train Acc: 87.44% | Val Loss: 0.3720 | Val Acc: 85.93%
  => Saved new best model checkpoint (V

In [42]:
# Pipeline 2 – Evaluate on test set
!python -m src.evaluation.evaluate --config configs/pipeline2_clahe.yaml --model_path outputs/pipeline2_clahe/best_model.pth

--- Evaluating model for: pipeline2_clahe ---
[test] Loading preprocessed dataset from: data/processed/clahe/test
Loaded 2755 images for split: test
Model weights loaded successfully.

--- Test Set Classification Report ---
              precision    recall  f1-score   support

         D00       0.80      0.66      0.72       248
         D20       0.82      0.81      0.81       307
         D40       0.86      0.90      0.88       492
      Normal       0.97      0.99      0.98      1708

    accuracy                           0.92      2755
   macro avg       0.86      0.84      0.85      2755
weighted avg       0.92      0.92      0.92      2755

Overall Accuracy: 92.38%

Evaluation completed. Metrics and confusion matrix saved to outputs/pipeline2_clahe
